## Import necessary libraries

In [4]:
import sys
import torch
import torch_geometric
import torch_scatter
import torch_sparse
import torch_cluster
import torch_spline_conv
import rdkit
import pandas as pd
import numpy as np
import scipy
import sklearn
import xgboost
import tqdm
import matplotlib
import seaborn
import streamlit
import requests

print("Success: All packages imported successfully!")
print(f"-> Using Python: {sys.executable}")
print(f"-> GPU Available: {torch.cuda.is_available()}")

Success: All packages imported successfully!
-> Using Python: /home/satyam-rana/miniconda3/envs/pytorch_env/bin/python
-> GPU Available: True


In [6]:
import os, json
import numpy as np

# Directory helpers
def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

def save_json(data, path: str):
    ensure_dir(os.path.dirname(path))
    with open(path, "w") as f:
        json.dump(data, f)

def load_json(path: str):
    with open(path, "r") as f:
        return json.load(f)

# STITCH ID helpers 
def normalize_stitch_id(sid: str) -> str:
    """Normalize any STITCH ID to uppercase stripped string."""
    return str(sid).strip().upper()

def stitch_to_pubchem_cid(sid: str) -> int:
    """Extract numeric PubChem CID from a STITCH ID (CIDs/CID1 prefix)."""
    sid = normalize_stitch_id(sid)
    if sid.startswith("CID1"):
        return int(sid[4:])
    if sid.startswith("CIDS"):
        return int(sid[4:])
    numeric = "".join(c for c in sid if c.isdigit())
    return int(numeric) if numeric else -1

def compute_pos_weight(labels: np.ndarray) -> float:
    """Return neg/pos ratio for BCEWithLogitsLoss pos_weight."""
    pos = labels.sum()
    neg = len(labels) - pos
    return neg / max(pos, 1)

# Directory setup 
DATA_DIR      = "data"
RAW_DIR       = os.path.join(DATA_DIR, "raw")
PROC_DIR      = os.path.join(DATA_DIR, "processed")
MAP_DIR       = os.path.join(DATA_DIR, "mappings")
MODELS_DIR    = "models"

for d in [RAW_DIR, PROC_DIR, MAP_DIR, MODELS_DIR]:
    ensure_dir(d)

print("Directories ready:", DATA_DIR, MODELS_DIR)

Directories ready: data models


## Load dataset

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

def _load_csv(path, sep=","):
    df = pd.read_csv(path, sep=sep, low_memory=False)
    print(f"  {os.path.basename(path)}: shape={df.shape}, cols={list(df.columns)}")
    nulls = df.isnull().sum().sum()
    if nulls:
        print(f"    WARNING: {nulls} null values")
    print(df.head(3).to_string(), "\n")
    return df

def load_all_data(data_dir: str = RAW_DIR) -> dict:
    """Load and normalize all 9 dataset files. Returns dict of DataFrames."""
    print("=" * 60)
    print("Loading datasets from:", data_dir)
    print("=" * 60)
    data = {}

    # bio-decagon-combo (4.6M rows - load in chunks)
    chunks = []
    for chunk in pd.read_csv(os.path.join(data_dir, "bio-decagon-combo.csv"),
                             chunksize=100_000, low_memory=False):
        chunk.columns = [c.strip() for c in chunk.columns]
        for col in chunk.columns:
            if "stitch" in col.lower():
                chunk[col] = chunk[col].apply(normalize_stitch_id)
        chunks.append(chunk)
    data["combo"] = pd.concat(chunks, ignore_index=True)
    print(f"  bio-decagon-combo: shape={data['combo'].shape}")
    print(data["combo"].head(3).to_string(), "\n")

    for fname, key, sep in [
        ("bio-decagon-mono.csv",              "mono",          ","),
        ("bio-decagon-ppi.csv",               "ppi",           ","),
        ("bio-decagon-targets.csv",           "targets",       ","),
        ("bio-decagon-targets-all.csv",       "targets_all",   ","),
        ("bio-decagon-effectcategories.csv",  "effectcategories", ","),
        ("meddra_all_se.tsv",                 "meddra_se",     "\t"),
        ("meddra_freq.tsv",                   "meddra_freq",   "\t"),
    ]:
        data[key] = _load_csv(os.path.join(data_dir, fname), sep=sep)
        for col in data[key].columns:
            if "stitch" in col.lower():
                data[key][col] = data[key][col].apply(normalize_stitch_id)

    data["chembl"] = pd.read_csv(os.path.join(data_dir, "chembl_37_chemreps.txt"),
                                  sep="\t", low_memory=False)
    data["chembl"].columns = [c.strip() for c in data["chembl"].columns]
    print(f"  chembl_37_chemreps: shape={data['chembl'].shape}")
    print(data["chembl"].head(3).to_string(), "\n")

    print("All datasets loaded.")
    return data

# Run 
data = load_all_data(RAW_DIR)

Loading datasets from: data/raw
  bio-decagon-combo: shape=(4649441, 4)
       STITCH 1      STITCH 2 Polypharmacy Side Effect            Side Effect Name
0  CID000002173  CID000003345                 C0151714             hypermagnesemia
1  CID000002173  CID000003345                 C0035344  retinopathy of prematurity
2  CID000002173  CID000003345                 C0004144                 atelectasis 

  bio-decagon-mono.csv: shape=(174977, 3), cols=['STITCH', 'Individual Side Effect', 'Side Effect Name']
         STITCH Individual Side Effect              Side Effect Name
0  CID003062316               C1096328   central nervous system mass
1  CID003062316               C0162830     Photosensitivity reaction
2  CID003062316               C1611725  leukaemic infiltration brain 

  bio-decagon-ppi.csv: shape=(715612, 2), cols=['Gene 1', 'Gene 2']
   Gene 1  Gene 2
0  114787  375519
1  114787  285613
2  114787    7448 

  bio-decagon-targets.csv: shape=(18690, 2), cols=['STITCH', 'Gene']


In [9]:
def build_id_mappings(data: dict, save_dir: str = MAP_DIR) -> dict:
    """
    Build drug/protein/SE index dicts and save as JSON.
    CRITICAL: protein indices are offset by num_drugs - they never overlap with drug indices.
    """
    ensure_dir(save_dir)
    combo, mono, targets, targets_all, meddra_se = (
        data["combo"], data["mono"], data["targets"],
        data["targets_all"], data["meddra_se"])

    def stitch_cols(df):
        return [c for c in df.columns if "stitch" in c.lower()]

    # Collect all drug STITCH IDs from every file that has them
    drug_ids = set()
    for df in [combo, mono, targets, targets_all, meddra_se]:
        for col in stitch_cols(df):
            drug_ids.update(df[col].dropna().unique())
    drug_ids = sorted(drug_ids)
    
    # FIX: Cast drug IDs to string keys for JSON compliance
    drug_to_idx = {str(d): int(i) for i, d in enumerate(drug_ids)}
    idx_to_drug = {str(i): str(d) for d, i in drug_to_idx.items()}
    num_drugs = len(drug_ids)

    # Protein nodes - offset by num_drugs (CRITICAL: no overlap with drug indices)
    protein_ids = set()
    for df in [data["ppi"], targets, targets_all]:
        for col in df.columns:
            if "gene" in col.lower() or "protein" in col.lower():
                protein_ids.update(df[col].dropna().unique())
    protein_ids = sorted(protein_ids)
    
    # FIX: Cast protein IDs to string keys (This addresses the exact traceback crash)
    protein_to_idx = {str(p): int(i + num_drugs) for i, p in enumerate(protein_ids)}
    num_proteins = len(protein_ids)

    # Side effect indices
    combo_cols = list(combo.columns)
    se_id_col   = combo_cols[2]
    se_name_col = combo_cols[3]
    se_ids = sorted(combo[se_id_col].dropna().unique())
    
    # FIX: Cast side effect targets to clear primitives
    se_to_idx  = {str(s): int(i) for i, s in enumerate(se_ids)}
    se_to_name = {str(k): str(v) for k, v in zip(combo[se_id_col], combo[se_name_col]) if pd.notna(k)}

    # Drug pair index (key = sorted tuple stored as "A|B")
    stitch1_col, stitch2_col = combo_cols[0], combo_cols[1]
    pair_keys = (combo[[stitch1_col, stitch2_col]]
                 .drop_duplicates()
                 .apply(lambda r: f"{min(r[stitch1_col], r[stitch2_col])}|"
                                  f"{max(r[stitch1_col], r[stitch2_col])}", axis=1)
                 .drop_duplicates().tolist())
    drug_pair_to_idx = {str(k): int(i) for i, k in enumerate(pair_keys)}

    print(f"Drugs: {num_drugs}, Proteins: {num_proteins}, "
          f"Side effects: {len(se_ids)}, Drug pairs: {len(pair_keys)}")

    save_json(drug_to_idx,     os.path.join(save_dir, "drug_to_idx.json"))
    save_json(idx_to_drug,     os.path.join(save_dir, "idx_to_drug.json"))
    save_json(protein_to_idx,  os.path.join(save_dir, "protein_to_idx.json"))
    save_json(se_to_idx,       os.path.join(save_dir, "se_to_idx.json"))
    save_json(se_to_name,      os.path.join(save_dir, "se_to_name.json"))
    save_json(drug_pair_to_idx, os.path.join(save_dir, "drug_pair_to_idx.json"))

    return dict(drug_to_idx=drug_to_idx, idx_to_drug=idx_to_drug,
                protein_to_idx=protein_to_idx, se_to_idx=se_to_idx,
                se_to_name=se_to_name, drug_pair_to_idx=drug_pair_to_idx,
                num_drugs=num_drugs, num_proteins=num_proteins,
                combo_stitch1=stitch1_col, combo_stitch2=stitch2_col,
                combo_se_id=se_id_col, combo_se_name=se_name_col)

# Run 
mappings = build_id_mappings(data, MAP_DIR)

Drugs: 2135, Proteins: 19122, Side effects: 1317, Drug pairs: 63473


In [10]:
def build_splits(drug_pair_to_idx: dict, harm_labels: np.ndarray,
                 save_path: str = os.path.join(PROC_DIR, "splits.npz")):
    """80/10/10 stratified split saved as splits.npz."""
    ensure_dir(os.path.dirname(save_path))
    indices = np.arange(len(drug_pair_to_idx))

    train_idx, temp_idx, y_train, y_temp = train_test_split(
        indices, harm_labels, test_size=0.2, stratify=harm_labels, random_state=42)
    val_idx, test_idx, _, _ = train_test_split(
        temp_idx, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

    for name, idx in [("train", train_idx), ("val", val_idx), ("test", test_idx)]:
        pos = harm_labels[idx].sum()
        print(f"  {name}: {len(idx)} pairs — harmful={int(pos)} ({100*pos/len(idx):.1f}%)")

    np.savez(save_path, train_idx=train_idx, val_idx=val_idx, test_idx=test_idx)
    print(f"Splits saved → {save_path}")
    return train_idx, val_idx, test_idx

In [12]:
import os
import sys
import time
import requests
import numpy as np
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

MORGAN_DIM = 2048
PUBCHEM_URL = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/property/IsomericSMILES/JSON"

def _smiles_from_chembl(cid: int, chembl_df) -> str | None:
    inchi_col  = next((c for c in chembl_df.columns if "inchi" in c.lower() and "key" not in c.lower()), None)
    smiles_col = next((c for c in chembl_df.columns if "smiles" in c.lower()), None)
    if inchi_col is None or smiles_col is None:
        return None
    mask = chembl_df[inchi_col].astype(str).str.contains(str(cid), na=False)
    hits = chembl_df.loc[mask, smiles_col].dropna()
    return hits.iloc[0] if len(hits) > 0 else None

def _smiles_from_pubchem(cid: int, retry: int = 2) -> str | None:
    for _ in range(retry):
        try:
            r = requests.get(PUBCHEM_URL.format(cid=cid), timeout=5)
            if r.status_code == 200:
                return r.json()["PropertyTable"]["Properties"][0].get("IsomericSMILES")
        except Exception:
            time.sleep(0.5)
    return None

def _morgan_fp(smiles: str) -> np.ndarray:
    try:
        from rdkit import Chem
        from rdkit.Chem import rdFingerprintGenerator
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return np.zeros(MORGAN_DIM, dtype=np.float32)
        # Using modern MorganGenerator API to remove deprecation warnings
        gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=MORGAN_DIM)
        fp = gen.GetFingerprintAsNumPy(mol)
        return fp.astype(np.float32)
    except Exception:
        return np.zeros(MORGAN_DIM, dtype=np.float32)

def build_drug_features(data: dict, drug_to_idx: dict,
                        save_path: str = os.path.join(PROC_DIR, "drug_features.npy")) -> np.ndarray:
    ensure_dir(os.path.dirname(save_path))
    chembl_df   = data["chembl"]
    mono_df     = data["mono"]
    meddra_se_df = data["meddra_se"]
    num_drugs   = len(drug_to_idx)

    # Build mono SE vocabulary
    mono_stitch_col  = next((c for c in mono_df.columns if "stitch" in c.lower()), mono_df.columns[0])
    mono_se_col      = mono_df.columns[1]
    meddra_stitch_col = next((c for c in meddra_se_df.columns if "stitch" in c.lower()), meddra_se_df.columns[0])
    meddra_se_col    = meddra_se_df.columns[-1]

    all_mono_ses = sorted(set(mono_df[mono_se_col].dropna()) | set(meddra_se_df[meddra_se_col].dropna()))
    mono_se_to_idx = {s: i for i, s in enumerate(all_mono_ses)}
    num_mono_se    = len(all_mono_ses)

    drug_mono = {d: set() for d in drug_to_idx}
    for _, row in mono_df.iterrows():
        if row[mono_stitch_col] in drug_mono:
            drug_mono[row[mono_stitch_col]].add(row[mono_se_col])
    for _, row in meddra_se_df.iterrows():
        if row[meddra_stitch_col] in drug_mono:
            drug_mono[row[meddra_stitch_col]].add(row[meddra_se_col])

    feature_dim   = MORGAN_DIM + num_mono_se
    drug_features = np.zeros((num_drugs, feature_dim), dtype=np.float32)

    # Worker function for parallelization
    def process_single_drug(item):
        drug_sid, idx = item
        cid = stitch_to_pubchem_cid(drug_sid)
        smiles = _smiles_from_chembl(cid, chembl_df)
        if smiles is None and cid > 0:
            smiles = _smiles_from_pubchem(cid)
        return idx, drug_sid, smiles

    print(f"Starting parallel processing with 20 workers for {num_drugs} drugs...")
    found_smiles = 0
    
    # Use max_workers=20 to request multiple smiles simultaneously
    with ThreadPoolExecutor(max_workers=20) as executor:
        futures = [executor.submit(process_single_drug, item) for item in drug_to_idx.items()]
        
        for future in tqdm(as_completed(futures), total=len(futures), desc="Drug features"):
            idx, drug_sid, smiles = future.result()
            if smiles is not None:
                drug_features[idx, :MORGAN_DIM] = _morgan_fp(smiles)
                found_smiles += 1
            for se in drug_mono.get(drug_sid, set()):
                se_i = mono_se_to_idx.get(se)
                if se_i is not None:
                    drug_features[idx, MORGAN_DIM + se_i] = 1.0

    print(f"\n{found_smiles}/{num_drugs} drugs matched to SMILES")
    print(f"Drug feature matrix shape: {drug_features.shape}")
    np.save(save_path, drug_features)
    return drug_features

# ── Run ────────────────────────────────────────────────────────────────────
drug_features = build_drug_features(data, mappings["drug_to_idx"])

Starting parallel processing with 20 workers for 2135 drugs...


Drug features: 100%|██████████| 2135/2135 [36:50<00:00,  1.04s/it] 


56/2135 drugs matched to SMILES
Drug feature matrix shape: (2135, 18355)
